# 

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Data2026mxmo.csv")
df.head()

,Match,Alliance,Robot1,Robot2,Robot3,totalAutoPoints,totalTeleopPoints,endGameRobot1,endGameRobot2,endGameRobot3,Score
0,1,red,6652,3933,6106,14,93,NaN,NaN,Level1,112
1,1,blue,3794,10931,7421,12,54,NaN,NaN,NaN,66
2,2,red,4723,3522,7102,5,22,NaN,NaN,NaN,37
3,2,blue,3478,9060,9282,2,81,NaN,NaN,NaN,88
4,3,red,6200,6832,9280,0,46,NaN,NaN,NaN,46


In [3]:
df.keys()

Index(['Match', 'Alliance', 'Robot1', 'Robot2', 'Robot3', 'totalAutoPoints',
       'totalTeleopPoints', 'endGameRobot1', 'endGameRobot2', 'endGameRobot3',
       'Score'],
      dtype='str')

In [6]:
import requests
import pandas as pd

TBA_KEY = "boEeyqrSMILCnYsNDP3zBuwPVT6IDJoNvneHDy5uw4el7hSWKSmPiTM5iPVX0Hxh"
HEADERS = {"X-TBA-Auth-Key": TBA_KEY}
EVENT = "2026mxmo"

rows = []

for i in range(1, 73):
    r = requests.get(f"https://www.thebluealliance.com/api/v3/match/{EVENT}_qm{i}", headers=HEADERS)
    
    if r.status_code != 200:
        print(f"qm{i} falló: {r.status_code}")
        continue
    
    match = r.json()
    sb = match.get("score_breakdown", {})
    alliances = match.get("alliances", {})

    for color in ["red", "blue"]:
        teams = alliances[color]["team_keys"]
        breakdown = sb.get(color, {})
        
        rows.append({
            "actual_time": match.get("actual_time"),
            "EventKey": EVENT,
            "Alliance": color,
            "Robot1": teams[0][3:] if len(teams) > 0 else None,
            "Robot2": teams[1][3:] if len(teams) > 1 else None,
            "Robot3": teams[2][3:] if len(teams) > 2 else None,
            "totalAutoPoints": breakdown.get("totalAutoPoints"),
            "totalTeleopPoints": breakdown.get("totalTeleopPoints"),
            "endGameRobot1": breakdown.get("endGameTowerRobot1"),
            "endGameRobot2": breakdown.get("endGameTowerRobot2"),
            "endGameRobot3": breakdown.get("endGameTowerRobot3"),
            "Score": breakdown.get("totalPoints")
        })

tba_df = pd.DataFrame(rows)
tba_df.head(10)

,actual_time,EventKey,Alliance,Robot1,Robot2,Robot3,totalAutoPoints,totalTeleopPoints,endGameRobot1,endGameRobot2,endGameRobot3,Score
0,1773414976,2026mxmo,red,6652,3933,6106,14,93,None,None,Level1,112
1,1773414976,2026mxmo,blue,3794,10931,7421,12,54,None,None,None,66
2,1773415541,2026mxmo,red,4723,3522,7102,5,22,None,None,None,37
3,1773415541,2026mxmo,blue,3478,9060,9282,2,81,None,None,None,88
4,1773416121,2026mxmo,red,6200,6832,9280,0,46,None,None,None,46
5,1773416121,2026mxmo,blue,4775,5932,9053,9,46,None,None,None,55
6,1773416712,2026mxmo,red,4010,6702,5133,4,23,None,None,None,32
7,1773416712,2026mxmo,blue,3472,10529,7546,2,40,None,None,None,52
8,1773417331,2026mxmo,red,3354,4371,9213,29,97,None,None,None,126
9,1773417331,2026mxmo,blue,8741,6606,11065,2,4,None,None,None,11


In [9]:
tba_df = df

In [2]:
import requests
import pandas as pd
import os
import datetime

TBA_KEY = "boEeyqrSMILCnYsNDP3zBuwPVT6IDJoNvneHDy5uw4el7hSWKSmPiTM5iPVX0Hxh"
HEADERS = {"X-TBA-Auth-Key": TBA_KEY}

os.makedirs("Data", exist_ok=True)

r = requests.get("https://www.thebluealliance.com/api/v3/events/2026/simple", headers=HEADERS)
events = r.json()

for event in events:
    event_key = event['key']
    output_path = f"Data/Data{event_key}.csv"

    # Saltar si ya existe
    if os.path.exists(output_path):
        print(f"Ya existe {event_key}, saltando...")
        continue

    print(f"Procesando {event_key}...")

    r_matches = requests.get(f"https://www.thebluealliance.com/api/v3/event/{event_key}/matches/simple", headers=HEADERS)
    if r_matches.status_code != 200:
        print(f"  Sin partidos para {event_key}, saltando...")
        continue
    
    qm_matches = [m for m in r_matches.json() if m['comp_level'] == 'qm']
    if not qm_matches:
        print(f"  Sin qualis para {event_key}, saltando...")
        continue

    max_match = max(m['match_number'] for m in qm_matches)
    rows = []

    for i in range(1, max_match + 1):
        r_match = requests.get(f"https://www.thebluealliance.com/api/v3/match/{event_key}_qm{i}", headers=HEADERS)
        if r_match.status_code != 200:
            print(f"  qm{i} falló: {r_match.status_code}")
            continue

        match = r_match.json()
        sb = match.get("score_breakdown")
        if not sb:
            continue
        alliances = match.get("alliances", {})

        for color in ["red", "blue"]:
            teams = alliances[color]["team_keys"]
            breakdown = sb.get(color, {})

            rows.append({
                "actual_time": datetime.datetime.fromtimestamp(match.get("actual_time"), datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S") if match.get("actual_time") else None,
                "EventKey": event_key,
                "Match": i,
                "Alliance": color,
                "Robot1": teams[0][3:] if len(teams) > 0 else None,
                "Robot2": teams[1][3:] if len(teams) > 1 else None,
                "Robot3": teams[2][3:] if len(teams) > 2 else None,
                "totalAutoPoints": breakdown.get("totalAutoPoints"),
                "totalTeleopPoints": breakdown.get("totalTeleopPoints"),
                "endGameRobot1": breakdown.get("endGameTowerRobot1"),
                "endGameRobot2": breakdown.get("endGameTowerRobot2"),
                "endGameRobot3": breakdown.get("endGameTowerRobot3"),
                "Score": breakdown.get("totalPoints")
            })

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"  Guardado: {output_path} ({max_match} partidos)")

print("¡Listo!")

Ya existe 2026alhu, saltando...
Ya existe 2026arc, saltando...
Ya existe 2026arli, saltando...
Ya existe 2026ausc, saltando...
Procesando 2026azdd...
  Sin qualis para 2026azdd, saltando...
Procesando 2026azddm1...
  Sin qualis para 2026azddm1, saltando...
Ya existe 2026azfg, saltando...
Ya existe 2026bcvi, saltando...
Ya existe 2026brba, saltando...
Ya existe 2026brsp, saltando...
Procesando 2026caab...
  Sin qualis para 2026caab, saltando...
Ya existe 2026caasv, saltando...
Ya existe 2026cacac, saltando...
Ya existe 2026caclv, saltando...
Ya existe 2026caetb, saltando...
Ya existe 2026cagle, saltando...
Ya existe 2026cahal, saltando...
Ya existe 2026calas, saltando...
Procesando 2026camou...
  Sin qualis para 2026camou, saltando...
Ya existe 2026cancmp, saltando...
Ya existe 2026caoec, saltando...
Ya existe 2026capin, saltando...
Ya existe 2026capoh, saltando...
Ya existe 2026casac, saltando...
Ya existe 2026cascmp, saltando...
Ya existe 2026casgv, saltando...
Ya existe 2026casnd, sa

In [1]:
import pandas as pd
import glob
import os

files = glob.glob("Data/*.csv")

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["source_file"] = os.path.basename(f)  # opcional, para trazabilidad
    dfs.append(df)

df_concatenado = pd.concat(dfs, ignore_index=True)
df_concatenado.to_csv("df_concatenado.csv", index=False)

print(f"Archivos concatenados: {len(dfs)}")
print(f"Shape final: {df_concatenado.shape}")

Archivos concatenados: 207
Shape final: (30090, 14)
